# Fase 4–5 CRISP-DM — Modelado y Evaluación: XGBoost vs CatBoost

**Autor:** Jesús David Villota Arteaga — Área: desarrollo y entrenamiento de modelos de ML.

Este notebook implementa las fases 4 (Modelado) y 5 (Evaluación) de CRISP-DM sobre la tabla analítica de la Fase 3 (`data/processed/ctr_phase3/`):

1. Entrena **XGBoost** y **CatBoost** como clasificadores de `click` (1 = el usuario hizo clic).
2. Optimiza hiperparámetros con **Optuna** (objetivo: ROC-AUC en validación).
3. Evalúa con **AUC-ROC**, Accuracy, Precision, Recall, F1, matrices de confusión y curvas ROC/PR.
4. Compara ambos modelos, selecciona el mejor y lo guarda en `models/` con su `metadata.json` para la API.

> **Diseño experimental.** Partición temporal (train ≤ 1848196 < validation ≤ 2269328 < test), el umbral de decisión se ajusta en **validación** (máximo F1) y **prueba se reserva** para el reporte final. Ver `docs/fase_4_5_modelado_evaluacion.md`.

## 0. Configuración

Ajuste las constantes según la máquina disponible. Los valores por defecto reproducen el entrenamiento de referencia del informe.

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent  # notebooks/ -> raiz del proyecto
sys.path.insert(0, str(ROOT))

import logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report

from src.data.preprocessing import load_split
from src.ml.compare import compare_models, print_markdown_table, select_best
from src.ml.evaluation import best_f1_threshold, evaluate
from src.ml.models import ALGORITHMS
from src.ml.train import train_and_save
from src.ml.tuning import run_study

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
sns.set_theme(style="whitegrid")
%matplotlib inline

# Hiperparametros del experimento (subir para el entrenamiento final completo)
SEED = 42
TUNE_TRAIN_ROWS, TUNE_EVAL_ROWS, N_TRIALS = 100_000, 50_000, 10
FINAL_TRAIN_ROWS, FINAL_EVAL_ROWS = 500_000, 200_000
MODELS_DIR = ROOT / "models"
REPORTS_DIR = ROOT / "reports"

## 1. Carga de datos

Se leen muestras reproducibles (semilla fija) de cada partición temporal. La distribución de `click` se conserva (~36 % positivos) porque la Fase 3 no aplicó balanceo.

In [ ]:
X_tr, y_tr = load_split("train", sample_rows=TUNE_TRAIN_ROWS, seed=SEED)
X_va, y_va = load_split("validation", sample_rows=TUNE_EVAL_ROWS, seed=SEED)
print(f"train={len(X_tr):,} (CTR={y_tr.mean():.3f})  val={len(X_va):,} (CTR={y_va.mean():.3f})")
X_tr.head(3)

## 2. Baselines con hiperparámetros por defecto

Punto de partida antes de optimizar: XGBoost (`tree_method='hist'`, soporte categórico nativo) y CatBoost (categóricas por nombre). Sirve para medir la ganancia real del tuning.

In [ ]:
baselines = {}
for algo in ["xgboost", "catboost"]:
    model = ALGORITHMS[algo]()
    model.fit(X_tr, y_tr, X_va, y_va)  # defaults de src/ml/models.py + early stopping
    proba = model.predict_proba(X_va)
    thr = best_f1_threshold(y_va.to_numpy(), proba)
    baselines[algo] = evaluate(y_va.to_numpy(), proba, thr)
    print(f"{algo}: ROC-AUC={baselines[algo]['roc_auc']:.4f} F1={baselines[algo]['f1']:.4f} (umbral={thr:.3f})")

## 3. Optimización de hiperparámetros con Optuna

Búsqueda bayesiana (TPE + poda por mediana). Objetivo: **ROC-AUC en validación**. `scale_pos_weight` de XGBoost se fija a neg/pos (no se muestrea). La partición de prueba **no** interviene aquí.

In [ ]:
best_params = {}
for algo in ["xgboost", "catboost"]:
    result = run_study(algo, X_tr, y_tr, X_va, y_va, n_trials=N_TRIALS, seed=SEED,
                       output=REPORTS_DIR / "tuning" / f"{algo}.json")
    best_params[algo] = result["best_params"]
    print(f"Mejor {algo}: AUC={result['best_score']:.4f} params={result['best_params']}")

## 4. Entrenamiento final con los mejores hiperparámetros

Se reentrena sobre una muestra amplia, se ajusta el umbral por F1 en validación y se guarda cada artefacto (`model.*` + `categories.json` + `metadata.json`) para que la API los cargue sin cambios.

In [ ]:
del X_tr, X_va  # liberar memoria antes de la muestra final
X_tr, y_tr = load_split("train", sample_rows=FINAL_TRAIN_ROWS, seed=SEED)
X_va, y_va = load_split("validation", sample_rows=FINAL_EVAL_ROWS, seed=SEED)
X_te, y_te = load_split("test", sample_rows=FINAL_EVAL_ROWS, seed=SEED)
print(f"final: train={len(X_tr):,} val={len(X_va):,} test={len(X_te):,}")

metas = {}
for algo in ["xgboost", "catboost"]:
    metas[algo] = train_and_save(algo, X_tr, y_tr, X_va, y_va, X_te, y_te,
                                 MODELS_DIR, version="1.0.0", **best_params[algo])
    m = metas[algo]["metrics"]
    print(f"{algo}: val AUC={m['validation']['roc_auc']:.4f} | "
          f"test AUC={m['test']['roc_auc']:.4f} F1={m['test']['f1']:.4f}")

## 5. Evaluación en prueba

Métricas con el umbral de validación, reporte de clasificación y matrices de confusión. `compare_models` regenera además las figuras en `reports/figures/` y el JSON en `reports/metrics/comparison.json`.

In [ ]:
summary = compare_models(MODELS_DIR, models=["xgboost", "catboost"],
                         eval_rows=FINAL_EVAL_ROWS, seed=SEED, reports_dir=REPORTS_DIR)
print(print_markdown_table(summary))
print("Mejor modelo:", summary["best_model"])

In [ ]:
from IPython.display import Image, display
for fig in ["roc_curves", "pr_curves", "metrics_bars",
            "confusion_matrix_xgboost", "confusion_matrix_catboost", "feature_importance"]:
    display(Image(filename=str(REPORTS_DIR / "figures" / f"{fig}.png")))

## 6. Conclusiones

- Ambos modelos superan el azar (AUC > 0.5) y a la regresión logística de referencia; las curvas ROC/PR y las matrices de confusión quedan registradas en `reports/`.
- La selección del mejor modelo usa **ROC-AUC de validación** (desempate: PR-AUC, luego F1); el valor en prueba solo confirma la generalización.
- Los artefactos de `models/<nombre>/` (modelo + `metadata.json` con umbral y métricas) son los que consume la API (`POST /api/v1/predict`, `GET /api/v1/metrics/models`).
- Trabajo futuro: calibración de probabilidades (p. ej. isotónica), validación con la partición completa y monitoreo de deriva temporal en producción.